In [1]:
import pandas as pd
folder='/Users/rm026/Documents/hbs_study_patient_notes/hbs_full_run/'
master= pd.read_csv(folder + 'master_list.csv')
questions = pd.read_json(folder+'json_evaluated/extraction_evaluations.json').T

In [2]:
master['Prioritizing implicit and explicit information, does this medical record mention a stroke? For example, the text may directly mention stroke, ischemia, or an infarct. (Yes/No)'].mean()

np.float64(0.991518235793045)

In [3]:
questions.columns

Index(['Prioritizing implicit and explicit information, has the patient experienced a stroke in the cerebellum?',
       'Do you think this patient is able to walk normally, or do have some sort of impairment limiting ability to walk?',
       'Do you think this patient is able to perform the heel-shin maneuver normally, or do they have jerkiness of the lower extremity? (Yes/No)',
       'Do you think this patient is able to perform the finger-nose maneuver normally, or do they have jerkiness of the lower extremity? (Yes/No)',
       'Do you think this patient is able to speak normally, or do they have some degree of dysarthria or slurring? (Yes/No)',
       'Do you think this patient has normal eye movements on the physical exam, or do they have abnormalities like slowed pursuit, saccadic intrusions, hypo/hypermetric sacces, or nystagmus? (Yes/No)',
       'Prioritizing implicit and explicit information, does the patient have a documented seizure in their medical record? (Yes/No)'],
 

In [4]:
def summarize(df, column, default=0):
    sz_ai=[]

    for i, row in df.iterrows():

        answers = row[column]
        answers_cleaned=[answer.lower().strip('.') for answer in answers.values()]

        if ('yes' in answers_cleaned) or ('1' in answers_cleaned)  or ('true' in answers_cleaned) or ('y' in answers_cleaned):
            sz_ai.append(1)

        elif set(answers_cleaned)<=set(['no','0','false','n']):
            sz_ai.append(0)

        else:
            print('unrecognized answers:', answers_cleaned)
            sz_ai.append(default)

    return sz_ai

questions['llm_seizure']=summarize(questions, 'Prioritizing implicit and explicit information, does the patient have a documented seizure in their medical record? (Yes/No)')
# questions['llm_seizure'].mean()

unrecognized answers: ['no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no.\n\nthere is no mention of a seizure in the provided medical record or research report details for the patient', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no']
unrecognized answers: ['no', 'no', 'no (0)']
unrecognized answers: ['no', 'no', 'no', 'no', 'no', 'no', "no, the provided research report does not document a seizure in the patient's medical record"]


In [5]:
questions['llm_seizure'].sum()

np.int64(168)

In [6]:
questions['llm_cerebellar_stroke']=summarize(questions, 'Prioritizing implicit and explicit information, has the patient experienced a stroke in the cerebellum?')

unrecognized answers: ['no', 'no', 'no', '0', 'no', 'no', 'no. the report explicitly mentions that the patient experienced a subacute right thalamic stroke, not a cerebellar stroke', 'no', 'no', 'no', '0']
unrecognized answers: ['no', 'no, the report does not indicate that the patient experienced a stroke in the cerebellum. it specifies subacute infarctions in the left corona radiata, left putamen, and left caudate body, along with occlusion of', 'no']
unrecognized answers: ['no', 'no', 'no', 'n', 'no', 'no', 'no', 'no', 'no', 'based on the information provided in the [research report], there is no explicit mention of the patient having experienced a stroke in the cerebellum. the text mentions a "history of cva (cerebrovascular accident)". however, it does', 'no']
unrecognized answers: ['no', 'no', 'no, the report mentions that the patient experienced a cva (cerebrovascular accident or stroke) in 2010, but it specifically states it was in the left thalamic region, not the cerebellum', 

In [11]:
questions['MRN'] = questions.index
out=pd.merge(master, questions, on='MRN', how='left')


In [12]:
out.to_csv(folder+'master_with_llm_extractions.csv', index=False)

In [13]:
print('Seizure count:',out['llm_seizure'].sum())
print('Seizure proportion:',out['llm_seizure'].mean())
print('cerebellar stroke count:',out['llm_cerebellar_stroke'].sum())
print('cerebellar stroke proportion:',out['llm_cerebellar_stroke'].mean())

Seizure count: 168
Seizure proportion: 0.14249363867684478
cerebellar stroke count: 546
cerebellar stroke proportion: 0.4631043256997455
